# 06 · Unsicherheit und Modellvergleich

Die Notebooks 02–05 berichten einzelne Zahlen wie 87,5 % oder 100 %. Dieses Notebook fragt, wie belastbar diese Zahlen sind:

1. **Konfidenzintervalle:** Was sagt „8 von 8 richtig“ wirklich aus?
2. **McNemar-Test:** Ist der Random Forest wirklich besser als die SVM?
3. **Bootstrap:** Konfidenzintervalle ohne Formel, auch für den Unterschied zwischen zwei Modellen.
4. **Permutation Importance:** Auf welche Merkmale stützt sich das Modell tatsächlich?

> **So arbeitest du mit dem Notebook**
> - Zellen mit `# TODO` füllst du selbst aus. Bis dahin lösen sie einen `NotImplementedError` aus.
> - Am Ende jedes Abschnitts steht eine **Prüfzelle**. Läuft sie ohne `AssertionError` durch, stimmt dein Code.
> - Die Zellen *Interpretation* sind absichtlich leer. Schreib sie in eigenen Worten: Genau darüber wirst du im Vorstellungsgespräch reden.
> - Voraussetzung ist `data/features.csv` (eine Zeile pro Text: `file`, `author` und die 12 Merkmale).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 42
FEATURES = [
    "AverageWordLength", "AverageSentenceLength", "AverageSyllables",
    "WienerSachtextFormel", "FRE_Ger", "TTR",
    "Avg !", "Avg ?", "Avg .", "Avg ,", "Avg ;", "Avg :",
]

df = (pd.read_csv(Path("../data/features.csv"))
        .rename(columns={"Author": "author"})
        .sort_values("file", ignore_index=True))
X, y = df[FEATURES], df["author"]
print(f"{len(df)} Texte:", y.value_counts().to_dict())

## 1 · Konfidenzintervalle für die Accuracy

Accuracy ist ein Anteil: *k* richtige Vorhersagen von *n* Texten. Behandeln wir jeden Text als unabhängigen Versuch, folgt *k* einer Binomialverteilung, und wir können ein 95-%-Konfidenzintervall um *k / n* legen.

Zwei gängige Verfahren:

- **Clopper-Pearson („exakt“)** arbeitet direkt mit der Binomialverteilung. Es ist konservativ: lieber zu breit als zu schmal.
- **Wilson** ist eine Näherung, die auch bei kleinem *n* und bei Ergebnissen nahe 0 % oder 100 % gut funktioniert. Das Lehrbuch-Intervall *p ± 1,96 · SE* versagt dort: Bei 8 von 8 ist der Standardfehler 0, es würde also [100 %, 100 %] behaupten.

### TODO 1a
Schreib `accuracy_ci(k, n, method)`, die das 95-%-Intervall als Tupel `(low, high)` zurückgibt.

*Tipp:* `binomtest(k, n).proportion_ci(confidence_level=0.95, method=...)` liefert ein Objekt mit `.low` und `.high`. Die Verfahren heißen `"exact"` und `"wilson"`.

In [ ]:
def accuracy_ci(k, n, method="wilson"):
    """95-%-Konfidenzintervall für eine Accuracy von k richtigen aus n."""
    # TODO 1a
    raise NotImplementedError("TODO 1a")

### TODO 1b (Zusatzaufgabe): Wilson von Hand
Setz das Wilson-Intervall selbst um, damit du weißt, was die Bibliothek rechnet. Mit $\hat p = k/n$ und $z = 1{,}96$:

$$\text{Mitte} = \frac{\hat p + \frac{z^2}{2n}}{1 + \frac{z^2}{n}} \qquad \text{halbe Breite} = \frac{z}{1 + \frac{z^2}{n}} \sqrt{\frac{\hat p (1-\hat p)}{n} + \frac{z^2}{4n^2}}$$

In [ ]:
def wilson_by_hand(k, n, z=1.96):
    """Wilson-Intervall nach der Formel oben."""
    # TODO 1b (Zusatzaufgabe)
    raise NotImplementedError("TODO 1b")

### TODO 1c
Im ursprünglichen Bericht wurde jedes Modell auf einem einzigen 80/20-Split mit 8 Testtexten bewertet. Bau ein DataFrame `report_table` mit den Spalten `model`, `correct`, `n`, `accuracy`, `ci_low`, `ci_high` und nimm dafür das **exakte** Verfahren.

In [ ]:
report = {"Decision Tree": 6, "Random Forest": 7, "SVM": 7, "Ensemble": 8}  # richtig von 8

# TODO 1c: report_table bauen
raise NotImplementedError("TODO 1c")

In [ ]:
# Prüfung Abschnitt 1
assert np.allclose(accuracy_ci(8, 8, "exact"), (0.6306, 1.0), atol=1e-3)
assert np.allclose(accuracy_ci(31, 36, "wilson"), (0.7134, 0.9392), atol=1e-3)
assert np.isclose(report_table.set_index("model").loc["Ensemble", "ci_low"], 0.6306, atol=1e-3)
try:
    assert np.allclose(wilson_by_hand(31, 36), accuracy_ci(31, 36, "wilson"), atol=1e-3)
    print("TODO 1b: bestanden")
except NotImplementedError:
    print("TODO 1b: übersprungen (Zusatzaufgabe)")
print("Abschnitt 1: alle Prüfungen bestanden")

### Interpretation 1
*Was bedeutet das Intervall für 8 von 8 für die Aussage „das Ensemble ordnet jeden Text richtig zu“? Lassen sich die vier Modelle mit diesen Intervallen überhaupt unterscheiden?*

…

## 2 · Gepaarter Vergleich: McNemar-Test

Random Forest und SVM werden auf **denselben** Texten getestet, ihre Ergebnisse sind also *gepaart*. Wer zwei getrennte Konfidenzintervalle vergleicht, verschenkt diese Information. Der McNemar-Test betrachtet nur die Texte, bei denen sich die beiden Modelle **uneinig** sind:

| | SVM richtig | SVM falsch |
|---|---|---|
| **RF richtig** | a | b |
| **RF falsch** | c | d |

Texte, die beide Modelle richtig (*a*) oder beide falsch (*d*) zuordnen, sagen nichts darüber, welches Modell besser ist. Wären beide gleich gut, wäre jede Uneinigkeit ein Münzwurf. Dann folgt *b* einer Binomialverteilung mit *n = b + c* und *p = 0,5*.

Wir verwenden Leave-one-out: Jeder Text wird genau einmal vorhergesagt, und zwar von einem Modell, das auf den anderen 35 Texten trainiert wurde.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=SEED)
svm = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1))

### TODO 2a
Erzeuge Leave-one-out-Vorhersagen für beide Modelle und mach daraus die booleschen NumPy-Arrays `rf_correct` und `svm_correct` (ein Eintrag pro Text, `True` = richtig).

*Tipp:* `cross_val_predict(model, X, y, cv=LeaveOneOut(), n_jobs=-1)`; vergleiche das Ergebnis mit `y.to_numpy()`.

In [ ]:
# TODO 2a: rf_pred, svm_pred, rf_correct und svm_correct definieren
raise NotImplementedError("TODO 2a")

### TODO 2b
1. Bau die 2 × 2-Tabelle als `table = np.array([[a, b], [c, d]])` (Zeilen: RF richtig / falsch, Spalten: SVM richtig / falsch).
2. Schreib `mcnemar_exact(b, c)`, die den zweiseitigen p-Wert zurückgibt. *Tipp:* `binomtest(b, b + c, 0.5).pvalue`. Was soll die Funktion zurückgeben, wenn sich die Modelle nie uneinig sind (`b + c == 0`)?

In [ ]:
def mcnemar_exact(b, c):
    """Zweiseitiger exakter McNemar-Test für b gegen c Uneinigkeiten."""
    # TODO 2b
    raise NotImplementedError("TODO 2b")

# TODO 2b: table aus rf_correct und svm_correct bauen

In [ ]:
# Prüfung Abschnitt 2
assert np.isclose(mcnemar_exact(2, 5), 0.453125)
assert np.isclose(mcnemar_exact(0, 10), 0.001953125)
assert mcnemar_exact(0, 0) == 1.0
assert table.sum() == len(y)
assert table[0, 1] == np.sum(rf_correct & ~svm_correct), "Zeilen = RF, Spalten = SVM"
print("Abschnitt 2: alle Prüfungen bestanden\n")

b, c = table[0, 1], table[1, 0]
print(f"RF {rf_correct.sum()}/{len(y)} vs. SVM {svm_correct.sum()}/{len(y)} richtig")
print(f"Nur RF richtig: {b}, nur SVM richtig: {c}, p = {mcnemar_exact(b, c):.3f}")

### TODO 2c
Ergänze eine kleine Tabelle mit der Leave-one-out-Accuracy beider Modelle und ihren 95-%-Wilson-Intervallen (nimm deine Funktion aus Abschnitt 1).

In [ ]:
# TODO 2c

### Interpretation 2
*Der Random Forest ordnet mehr Texte richtig zu als die SVM. Was sagt der Test über diesen Unterschied? Warum beweist ein nicht signifikantes Ergebnis nicht, dass beide Modelle gleich gut sind?*

…

## 3 · Bootstrap: Konfidenzintervalle ohne Formel

Der Bootstrap schätzt die Unsicherheit durch wiederholtes Ziehen: Zieh *n* Ergebnisse **mit Zurücklegen** aus deinen *n* Ergebnissen, berechne die Kennzahl, wiederhole das tausendfach und nimm das 2,5- und das 97,5-Perzentil. Das klappt auch für Kennzahlen ohne einfache Formel, zum Beispiel für den *Unterschied* zwischen zwei Modellen.

> **Zu `evaluate_cv` im SVM-Notebook:** Die Funktion zieht den *Datensatz* neu und führt auf dieser Stichprobe eine Kreuzvalidierung durch. Beim Ziehen mit Zurücklegen entstehen Duplikate, also kann derselbe Text gleichzeitig in einem Trainings- und einem Test-Fold landen. Die Ergebnisse sehen dann besser aus, als sie sind. Hier ziehen wir stattdessen die *Ergebnisse pro Text* neu.

### TODO 3a
Schreib `bootstrap_ci(correct, n_boot=10_000, seed=SEED)`: Zieh aus dem booleschen Array `correct` neue Stichproben, nimm von jeder den Mittelwert und gib das 2,5- und das 97,5-Perzentil zurück.

*Tipp:* `rng = np.random.default_rng(seed)`, dann `rng.choice(correct, size=(n_boot, len(correct)), replace=True).mean(axis=1)` und `np.percentile(..., [2.5, 97.5])`.

In [ ]:
def bootstrap_ci(correct, n_boot=10_000, seed=SEED):
    """Perzentil-Bootstrap-Intervall für den Anteil richtiger Vorhersagen."""
    # TODO 3a
    raise NotImplementedError("TODO 3a")

### TODO 3b
Bootstrappe den **Unterschied** der Accuracy zweier Modelle. Zieh in jeder Runde für beide Modelle *dieselben* Texte (Indizes ziehen, nicht Werte), damit die Paarung erhalten bleibt. Gib das 95-%-Intervall von `accuracy_a - accuracy_b` zurück.

In [ ]:
def bootstrap_diff_ci(correct_a, correct_b, n_boot=10_000, seed=SEED):
    """Gepaartes Perzentil-Bootstrap-Intervall für accuracy_a - accuracy_b."""
    # TODO 3b
    raise NotImplementedError("TODO 3b")

In [ ]:
# Prüfung Abschnitt 3
known = np.array([True] * 31 + [False] * 5)
low, high = bootstrap_ci(known)
assert 0.69 <= low <= 0.78 and 0.94 <= high <= 1.0, (low, high)
assert np.allclose(bootstrap_ci(known), bootstrap_ci(known)), "Seed verwenden: Ergebnisse müssen reproduzierbar sein"
assert np.allclose(bootstrap_diff_ci(known, known), (0.0, 0.0)), "für beide Modelle dieselben Texte ziehen"
print("Abschnitt 3: alle Prüfungen bestanden\n")

print("Accuracy Random Forest:", np.round(bootstrap_ci(rf_correct), 3))
print("Accuracy SVM:          ", np.round(bootstrap_ci(svm_correct), 3))
print("Unterschied RF - SVM:  ", np.round(bootstrap_diff_ci(rf_correct, svm_correct), 3))

### Interpretation 3
*Vergleiche das Bootstrap-Intervall des Random Forest mit seinem Wilson-Intervall aus Abschnitt 2. Enthält das Intervall für den Unterschied RF − SVM die 0, und passt das zum McNemar-Test?*

…

## 4 · Auf welche Merkmale stützt sich das Modell wirklich?

`RandomForestClassifier.feature_importances_` (die *Impurity Importance*) wird auf den **Trainingsdaten** berechnet und bevorzugt Merkmale mit vielen verschiedenen Werten. Die **Permutation Importance** misst direkter: Wie stark sinkt die Accuracy auf **ungesehenen** Texten, wenn man die Werte eines Merkmals zufällig vertauscht?

In [ ]:
rf_full = RandomForestClassifier(n_estimators=300, random_state=SEED).fit(X, y)
impurity = pd.Series(rf_full.feature_importances_, index=FEATURES, name="impurity")
impurity.sort_values(ascending=False).round(3)

### TODO 4a
Berechne die Permutation Importance auf zurückgehaltenen Folds:

1. Lauf über `StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y)`.
2. Trainiere auf dem Trainingsteil jeweils einen neuen `RandomForestClassifier(n_estimators=300, random_state=SEED)`.
3. Ruf `permutation_importance(model, X_test, y_test, n_repeats=30, random_state=SEED, n_jobs=-1)` auf und behalte `.importances_mean`.
4. Mittle über die Folds: `perm = pd.Series(..., index=FEATURES, name="permutation")`.

In [ ]:
# TODO 4a: perm definieren
raise NotImplementedError("TODO 4a")

In [ ]:
# Prüfung Abschnitt 4
assert isinstance(perm, pd.Series) and list(perm.index) == FEATURES
assert perm.notna().all()
print("Abschnitt 4: alle Prüfungen bestanden")


def plot_importances(a, b):
    """Zwei liegende Balkendiagramme nebeneinander, beide nach der ersten Reihe sortiert."""
    order = a.sort_values().index
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
    for ax, s in zip(axes, (a, b)):
        ax.barh(order, s[order], color="#2a78d6", height=0.6)
        ax.axvline(0, color="#c3c2b7", linewidth=1)
        ax.set_title(f"{s.name.capitalize()} Importance", loc="left")
        ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    return fig


plot_importances(impurity, perm);

### Interpretation 4
*Welche Merkmale behalten ihre Wichtigkeit, welche verlieren sie? FRE wird aus Satzlänge und Silben pro Wort berechnet. Was passiert mit der Permutation Importance eines Merkmals, wenn dieselbe Information noch in einem korrelierten Merkmal steckt?*

…

## 5 · Zusammenfassung für das README

Ausfüllen, sobald alle Prüfungen bestehen:

| | Ergebnis |
|---|---|
| Ensemble auf dem ursprünglichen Split mit 8 Texten | 8/8 richtig, 95-%-KI … |
| Random Forest, Leave-one-out | …/36, 95-%-KI … |
| SVM, Leave-one-out | …/36, 95-%-KI … |
| RF vs. SVM, McNemar | p = … |
| Wichtigstes Merkmal (Permutation) | … |

**Das solltest du erklären können:**
- Warum ist „die Konfidenzintervalle überlappen sich“ kein guter Weg, zwei Modelle auf denselben Daten zu vergleichen?
- Was bedeutet ein p-Wert von 0,45, und was bedeutet er *nicht*?
- Worin unterscheidet sich ein Bootstrap-Intervall von einem Wilson-Intervall, und wann nimmst du welches?
- Warum berechnet man die Permutation Importance auf zurückgehaltenen Daten?